In [6]:
!pip install \
  torch_geometric \
  pyg_lib torch_scatter \
  torch_sparse \
  torch_cluster \
  torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install imbalanced-learn \
  tqdm

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html


In [7]:
import os
import time
import pickle
import torch
import numpy as np
from torch.optim import Adam
from sklearn.metrics import f1_score, precision_score, recall_score
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv, BatchNorm
from google.colab import drive

# Initialize Google Drive connection
drive.mount('/content/drive', force_remount=True)

# Configure paths
DRIVE_PATH = '/content/drive/MyDrive/CyberThreatDetectionSystem_Project/'
DATA_PATH = os.path.join(DRIVE_PATH, 'Data/processed/')
MODEL_PATH = os.path.join(DRIVE_PATH, 'Models/')
os.makedirs(MODEL_PATH, exist_ok=True)

def load_graph_data():
    """Load train, validation, and test graphs from disk."""
    def _load(split):
        with open(os.path.join(DATA_PATH, f'{split}_graph.pkl'), 'rb') as f:
            return pickle.load(f)
    return _load('train'), _load('val'), _load('test')

def process_graph_data(raw_train, raw_val, raw_test):
    """Apply feature scaling matching original training setup."""
    train_feats = torch.tensor(raw_train['x'], dtype=torch.float32)
    q_low, q_high = torch.quantile(train_feats, torch.tensor([0.01, 0.99]), dim=0)
    iqr = torch.where(q_high - q_low > 1e-6, q_high - q_low, torch.ones_like(q_low))

    def _to_data(obj):
        x_scaled = torch.clamp((torch.tensor(obj['x'], dtype=torch.float32) - q_low) / iqr,
                               min=-5.0, max=5.0)
        return Data(
            x=x_scaled,
            edge_index=torch.tensor(obj['edge_index'], dtype=torch.long),
            y=torch.tensor(obj['y'], dtype=torch.long)
        )
    return _to_data(raw_train), _to_data(raw_val), _to_data(raw_test)

def create_loaders(train, val, test, batch_size=4096):
    """Create data loaders with consistent neighbor sampling."""
    num_neighbors = [50, 40, 30, 20]
    return (
        NeighborLoader(train, num_neighbors=num_neighbors, batch_size=batch_size, shuffle=True),
        NeighborLoader(val, num_neighbors=num_neighbors, batch_size=batch_size, shuffle=False),
        NeighborLoader(test, num_neighbors=num_neighbors, batch_size=batch_size, shuffle=False)
    )

class CyberThreatDetector(torch.nn.Module):
    def __init__(self, in_channels, hidden=192, dropout=0.20, n_layers=4):
        super().__init__()
        self.input_proj = torch.nn.Linear(in_channels, hidden)
        self.input_norm = torch.nn.LayerNorm(hidden)

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.lns = torch.nn.ModuleList()

        for _ in range(n_layers):
            self.convs.append(SAGEConv(hidden, hidden, aggr="mean"))
            self.bns.append(BatchNorm(hidden))
            self.lns.append(torch.nn.LayerNorm(hidden))

        self.attention = torch.nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )

        self.fc1 = torch.nn.Linear(hidden, hidden // 2)
        self.output = torch.nn.Linear(hidden // 2, 1)
        self.dropout = torch.nn.Dropout(dropout)
        self.gelu = torch.nn.GELU()
        self.n_layers = n_layers

    def forward(self, x, edge_index):
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = self.gelu(x)

        for i in range(self.n_layers):
            identity = x
            x = self.lns[i](x)
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = self.gelu(x)
            x = self.dropout(x)
            x = x + identity

        x_reshaped = x.unsqueeze(1)
        x_attn, _ = self.attention(x_reshaped, x_reshaped, x_reshaped)
        x = x + x_attn.squeeze(1)

        x = self.fc1(x)
        x = self.gelu(x)
        x = self.dropout(x)
        return self.output(x).squeeze()

def quantize_dynamic_model(model: torch.nn.Module) -> torch.nn.Module:
    """Quantize model weights to INT8 for efficiency."""
    return torch.quantization.quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )

def evaluate_model(model, loader, threshold, device='cpu'):
    """Calculate detection metrics using specified threshold."""
    model.eval()
    probs, truths = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index)[:batch.batch_size]
            probs.append(torch.sigmoid(logits).cpu())
            truths.append(batch.y[:batch.batch_size].cpu())
    probs = torch.cat(probs).numpy()
    truths = torch.cat(truths).numpy()
    preds = probs > threshold
    return {
        'f1': f1_score(truths, preds),
        'precision': precision_score(truths, preds),
        'recall': recall_score(truths, preds)
    }

def remap_legacy_keys(state):
    """Update model keys from old format to new ModuleList structure."""
    if any(k.startswith('convs.') for k in state):
        return state

    mapping = {}
    for block_idx in range(4):
        old_prefix = f'conv{block_idx+1}'
        new_prefix = f'convs.{block_idx}'
        mapping.update({k: k.replace(old_prefix, new_prefix, 1)
                        for k in state if k.startswith(old_prefix)})

        old_prefix = f'bn{block_idx+1}'
        new_prefix = f'bns.{block_idx}'
        mapping.update({k: k.replace(old_prefix, new_prefix, 1)
                        for k in state if k.startswith(old_prefix)})

        old_prefix = f'ln{block_idx+1}'
        new_prefix = f'lns.{block_idx}'
        mapping.update({k: k.replace(old_prefix, new_prefix, 1)
                        for k in state if k.startswith(old_prefix)})

    for old_k, new_k in mapping.items():
        state[new_k] = state.pop(old_k)

    return state

def run_ptq_comparison(ckpt_paths):
    """Compare performance between original and quantized models."""
    raw_train, raw_val, raw_test = load_graph_data()
    train_data, val_data, test_data = process_graph_data(raw_train, raw_val, raw_test)
    train_loader, val_loader, test_loader = create_loaders(
        train_data, val_data, test_data, batch_size=4096
    )

    for ckpt_path in ckpt_paths:
        print(f"\nProcessing {os.path.basename(ckpt_path)}")

        checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
        state_dict = remap_legacy_keys(state_dict)
        threshold = checkpoint.get('threshold', 0.115)

        in_feats = state_dict['input_proj.weight'].shape[1]
        hidden = state_dict['input_proj.weight'].shape[0]
        model_fp32 = CyberThreatDetector(in_channels=in_feats, hidden=hidden)
        model_fp32.load_state_dict(state_dict, strict=True)

        model_int8 = quantize_dynamic_model(model_fp32)

        t0 = time.time()
        metrics_fp32 = evaluate_model(model_fp32, test_loader, threshold)
        time_fp32 = time.time() - t0

        t0 = time.time()
        metrics_int8 = evaluate_model(model_int8, test_loader, threshold)
        time_int8 = time.time() - t0

        int8_path = ckpt_path.replace('.pth', '_INT8.pt')
        torch.save(model_int8.state_dict(), int8_path)

        print(f"Saved quantized model -> {os.path.basename(int8_path)}")
        print(f"* Accuracy (threshold {threshold:.3f})")
        for k in ('f1', 'precision', 'recall'):
            print(f"  {k:<9}: FP32 = {metrics_fp32[k]:.6f}   INT8 = {metrics_int8[k]:.6f}")
        print(f"* CPU time : FP32 {time_fp32:.2f}s   INT8 {time_int8:.2f}s")
        print(f"* File size: FP32 {os.path.getsize(ckpt_path)/1e6:.2f} MB   "
              f"INT8 {os.path.getsize(int8_path)/1e6:.2f} MB")

if __name__ == "__main__":
    checkpoints = [
        os.path.join(MODEL_PATH, f'model_{i}.pth') for i in range(10)
    ] + [os.path.join(MODEL_PATH, 'best_model.pth')]

    run_ptq_comparison(checkpoints)

Mounted at /content/drive

Processing model_0.pth
Saved quantized model -> model_0_INT8.pt
* Accuracy (threshold 0.115)
  f1       : FP32 = 0.957999   INT8 = 0.958341
  precision: FP32 = 0.938535   INT8 = 0.939026
  recall   : FP32 = 0.978288   INT8 = 0.978466
* CPU time : FP32 0.06s   INT8 0.06s
* File size: FP32 1.90 MB   INT8 1.84 MB

Processing model_1.pth
Saved quantized model -> model_1_INT8.pt
* Accuracy (threshold 0.115)
  f1       : FP32 = 0.910026   INT8 = 0.912844
  precision: FP32 = 0.958999   INT8 = 0.960676
  recall   : FP32 = 0.865812   INT8 = 0.869550
* CPU time : FP32 0.06s   INT8 0.06s
* File size: FP32 1.90 MB   INT8 1.84 MB

Processing model_2.pth
Saved quantized model -> model_2_INT8.pt
* Accuracy (threshold 0.115)
  f1       : FP32 = 0.948805   INT8 = 0.949001
  precision: FP32 = 0.933368   INT8 = 0.934749
  recall   : FP32 = 0.964762   INT8 = 0.963695
* CPU time : FP32 0.06s   INT8 0.06s
* File size: FP32 1.90 MB   INT8 1.84 MB

Processing model_3.pth
Saved quant